# 🎧 EPUB → Audiolibro con VoxCPM2

Convierte un libro **EPUB** en audiolibro (un **MP3 por capítulo** + un **.m4b** con marcadores) usando la voz de **VoxCPM2**. Funciona en **Google Colab** y en **Kaggle**, gratis.

### En Google Colab
1. Menú **Entorno de ejecución → Cambiar tipo de entorno → GPU T4**.
2. En tu Google Drive crea la carpeta **`Audiolibro`** y pon ahí tu archivo `.epub`. Si no lo encuentra, el notebook te pedirá subirlo.
3. **Entorno de ejecución → Ejecutar todo**. Te pedirá permiso para usar tu Drive.
4. Los audios quedan en **Drive → Audiolibro → (nombre del libro)**. Si Colab se desconecta, vuelve a **Ejecutar todo** y sigue donde quedó.

### En Kaggle
1. Crea un notebook nuevo e importa este archivo (**File → Import Notebook**).
2. Panel derecho: **Accelerator → GPU T4 x2** e **Internet → On**. Para esto Kaggle pide verificar tu cuenta con un número de teléfono.
3. **Add Input → Upload** y sube tu `.epub` (y tu audio de voz, si vas a clonar una).
4. Para la primera prueba: **Run All**. Para el libro completo, pon `PRUEBA = False` y usa **Save Version → Save & Run All**: corre en segundo plano aunque cierres el navegador (máx. 12 h).
5. Los archivos quedan en la pestaña **Output** de esa versión.
   *Si el libro no alcanzó a terminar:* agrega el Output de la versión anterior como Input (**Add Input → Your Work**) y vuelve a correr. Retoma desde ahí.

> 💡 **Primero haz una prueba** (`PRUEBA = True`): genera solo unos segundos para que escuches la voz. Si te gusta, cambia a `False`.


In [ ]:
#@title ⚙️ Configuración { display-mode: "form" }
# ---------- Cambia solo esta celda ----------

PRUEBA = True  #@param {type:"boolean"}
# True  = genera una muestra corta para escuchar la voz
# False = genera el libro completo

ARCHIVO_EPUB = ""  #@param {type:"string"}
# Nombre del .epub (ej: "mi_libro.epub"). Vacío = usa el que encuentre.

MODO_VOZ = "diseñar"  #@param ["diseñar", "clonar"]
# diseñar = crea la voz a partir de la descripción de abajo
# clonar  = imita la voz de un audio tuyo (10 a 20 segundos, claro y sin música)

DESCRIPCION_VOZ = "A warm, calm female narrator in her 30s, clear Latin American Spanish accent, moderate pace, soothing audiobook narration"  #@param {type:"string"}
# Funciona mejor en inglés: edad, género, tono, ritmo, emoción.

AUDIO_PARA_CLONAR = ""  #@param {type:"string"}
# Solo si MODO_VOZ = "clonar". Nombre del archivo (ej: "mi_voz.wav"), en la misma carpeta que el EPUB.

CAPITULOS = ""  #@param {type:"string"}
# Vacío = todos. O una lista: "1,2,5" o un rango: "3-7". "primero" = solo el primer capítulo. "siguiente" = el próximo capítulo que falta

CALIDAD = 10  #@param {type:"slider", min:6, max:20, step:2}
# Más alto = un poco mejor, pero más lento. 10 es un buen equilibrio.

UN_SOLO_ARCHIVO_M4B = True  #@param {type:"boolean"}
# Además de los MP3 por capítulo, crea un .m4b con marcadores de capítulo.

HORAS_MAXIMAS = 11.5  #@param {type:"number"}
# Se detiene ordenadamente antes de este tiempo (Colab y Kaggle cortan a las 12 h).


In [ ]:
#@title 📦 Instalar (tarda 1–3 minutos)
!pip install -q voxcpm ebooklib beautifulsoup4 soundfile
!ffmpeg -version | head -1


In [ ]:
#@title 📁 Buscar el libro y preparar carpetas
import os, sys, glob
# Oculta las barras de progreso internas de VoxCPM (una por frase): llenaban el log
# y dejaban el navegador lentísimo en Kaggle. El avance del libro se sigue viendo.
os.environ["TQDM_DISABLE"] = "1"

EN_KAGGLE = os.path.exists("/kaggle/working")
EN_COLAB = not EN_KAGGLE and ("google.colab" in sys.modules or os.path.exists("/content"))

if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    CARPETA = "/content/drive/MyDrive/Audiolibro"
    os.makedirs(CARPETA, exist_ok=True)
    BUSCAR_EN = [CARPETA]
    BASE_SALIDA = CARPETA
elif EN_KAGGLE:
    BUSCAR_EN = ["/kaggle/input"]
    BASE_SALIDA = "/kaggle/working"
else:  # tu propio computador
    BUSCAR_EN = ["."]
    BASE_SALIDA = "./Audiolibro"

def buscar(extensiones, nombre=""):
    hallados = []
    for raiz in BUSCAR_EN:
        for ext in extensiones:
            hallados += glob.glob(os.path.join(raiz, "**", "*" + ext), recursive=True)
    hallados = [h for h in hallados if "/_partes/" not in h]
    if nombre:
        hallados = [h for h in hallados if os.path.basename(h).lower() == nombre.lower()]
    return sorted(hallados, key=os.path.getmtime, reverse=True)

epubs = buscar([".epub"], ARCHIVO_EPUB)
if not epubs and EN_COLAB:
    from google.colab import files
    print("No encontré un .epub en Drive/Audiolibro. Súbelo aquí:")
    for nombre, datos in files.upload().items():
        with open(os.path.join(CARPETA, nombre), "wb") as f:
            f.write(datos)
    epubs = buscar([".epub"], ARCHIVO_EPUB)
if not epubs:
    raise FileNotFoundError("No encontré ningún .epub. Revisa las instrucciones de arriba.")
if len(epubs) > 1:
    print("Encontré varios EPUB; uso el más reciente. Para elegir otro, escribe su nombre en ARCHIVO_EPUB:")
    for e in epubs: print("  ·", os.path.basename(e))
RUTA_EPUB = epubs[0]
print("📖 Libro:", os.path.basename(RUTA_EPUB))

RUTA_CLON = None
if MODO_VOZ == "clonar":
    if not AUDIO_PARA_CLONAR:
        raise ValueError("Elegiste clonar: escribe el nombre del audio en AUDIO_PARA_CLONAR.")
    audios = buscar([".wav", ".mp3", ".m4a", ".flac", ".ogg"], AUDIO_PARA_CLONAR)
    if not audios:
        raise FileNotFoundError(f"No encontré el audio '{AUDIO_PARA_CLONAR}' junto al EPUB.")
    RUTA_CLON = audios[0]
    print("🎙️ Voz a clonar:", os.path.basename(RUTA_CLON))


In [ ]:
#@title 🔧 Funciones para leer el EPUB
# ===== Lectura del EPUB y preparación del texto =====
import re, os, glob, json, subprocess, shutil, unicodedata
import warnings
import ebooklib
from ebooklib import epub
from bs4 import BeautifulSoup

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

BLOQUES = ["h1", "h2", "h3", "h4", "p", "li", "blockquote", "div"]


def _limpiar(t):
    t = unicodedata.normalize("NFC", t)
    t = t.replace("­", "")              # guiones invisibles
    t = re.sub(r"[ \t ]+", " ", t)
    return t.strip()


def leer_epub(ruta, minimo_caracteres=300):
    """Devuelve una lista de capítulos: [{'titulo': str, 'texto': str}, ...]
    en el orden de lectura del libro. Omite portada, índice y páginas casi vacías."""
    libro = epub.read_epub(ruta)
    titulo_libro = (libro.get_metadata("DC", "title") or [["Libro"]])[0][0]
    capitulos = []
    for idref, _ in libro.spine:
        item = libro.get_item_with_id(idref)
        if item is None or item.get_type() != ebooklib.ITEM_DOCUMENT:
            continue
        sopa = BeautifulSoup(item.get_content(), "html.parser")
        for basura in sopa(["script", "style", "nav", "sup"]):
            basura.decompose()
        cuerpo = sopa.body or sopa
        parrafos = []
        for el in cuerpo.find_all(BLOQUES):
            # solo bloques "hoja" (sin otros bloques dentro) para no duplicar texto
            if el.find(BLOQUES):
                continue
            t = _limpiar(el.get_text(" "))
            if t:
                parrafos.append(t)
        if not parrafos:
            t = _limpiar(cuerpo.get_text(" "))
            parrafos = [t] if t else []
        texto = "\n".join(parrafos)
        if len(texto) < minimo_caracteres:
            continue
        cab = cuerpo.find(["h1", "h2", "h3"])
        titulo = _limpiar(cab.get_text(" ")) if cab else ""
        capitulos.append({"titulo": titulo or f"Capítulo {len(capitulos)+1}", "texto": texto})
    return titulo_libro, capitulos


_FIN_FRASE = re.compile(r"(?<=[\.\!\?…»\"”])\s+(?=[¿¡«\"“A-ZÁÉÍÓÚÑ0-9—–-])")


def trocear(texto, max_chars=250):
    """Divide el texto en fragmentos cortos (frases completas) para el modelo."""
    fragmentos = []
    actual = ""
    for parrafo in texto.split("\n"):
        parrafo = parrafo.strip()
        # títulos y líneas sin puntuación final: agregar punto para que haga la pausa
        if parrafo and not re.search(r"[\.\!\?…:;»\"”\)]$", parrafo):
            parrafo += "."
        frases = [f.strip() for f in _FIN_FRASE.split(parrafo) if f.strip()]
        for frase in frases:
            # frases muy largas: cortar por comas / punto y coma
            if len(frase) > max_chars:
                piezas = re.split(r"(?<=[,;:])\s+", frase)
            else:
                piezas = [frase]
            for p in piezas:
                while len(p) > max_chars:          # último recurso: por espacios
                    corte = p.rfind(" ", 0, max_chars)
                    corte = corte if corte > 0 else max_chars
                    if actual:
                        fragmentos.append(actual); actual = ""
                    fragmentos.append(p[:corte].strip())
                    p = p[corte:].strip()
                if not p:
                    continue
                if len(actual) + len(p) + 1 <= max_chars:
                    actual = (actual + " " + p).strip()
                else:
                    if actual:
                        fragmentos.append(actual)
                    actual = p
    if actual:
        fragmentos.append(actual)
    # el modelo no necesita fragmentos sin letras ni números
    return [f for f in fragmentos if re.search(r"\w", f)]


def nombre_archivo(s, largo=60):
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
    s = re.sub(r"[^\w\s-]", "", s).strip()
    s = re.sub(r"[\s]+", "_", s)
    return (s[:largo] or "sin_titulo")

TITULO_LIBRO, CAPS = leer_epub(RUTA_EPUB)
SALIDA = os.path.join(BASE_SALIDA, nombre_archivo(TITULO_LIBRO))
os.makedirs(SALIDA, exist_ok=True)

def _parse_caps(s, n):
    if not s.strip():
        return None
    if s.strip().lower() == "siguiente":
        return None  # se decide más abajo, después de recuperar lo ya hecho
    if s.strip().lower() == "primero":
        # el primer capítulo "de verdad" (salta portadas y dedicatorias cortas), con los anteriores
        elegidos = set()
        for i, c in enumerate(CAPS, 1):
            elegidos.add(i)
            if len(c["texto"]) >= 1500:
                break
        return elegidos
    elegidos = set()
    for parte in s.replace(" ", "").split(","):
        if "-" in parte:
            a, b = parte.split("-"); elegidos.update(range(int(a), int(b) + 1))
        elif parte:
            elegidos.add(int(parte))
    return {c for c in elegidos if 1 <= c <= n}

SOLO = _parse_caps(CAPITULOS, len(CAPS))
print(f"«{TITULO_LIBRO}» · {len(CAPS)} capítulos · {sum(len(c['texto']) for c in CAPS):,} caracteres")
for i, c in enumerate(CAPS, 1):
    marca = "" if (SOLO is None or i in SOLO) else "  (omitido)"
    print(f"  {i:3d}. {c['titulo'][:60]}  —  {len(c['texto']):,} car.{marca}")
print("\n💾 Los audios se guardarán en:", SALIDA)

# Recuperar capítulos ya hechos en una corrida anterior (Kaggle: Output agregado como Input)
if EN_KAGGLE:
    import shutil
    for viejo in glob.glob("/kaggle/input/**/*.mp3", recursive=True) + \
                 glob.glob("/kaggle/input/**/_partes/**/*.wav", recursive=True) + \
                 glob.glob("/kaggle/input/**/*.json", recursive=True) + \
                 glob.glob("/kaggle/input/**/voz_referencia.wav", recursive=True):
        rel = viejo.split(nombre_archivo(TITULO_LIBRO) + "/", 1)
        if len(rel) == 2:
            destino = os.path.join(SALIDA, rel[1])
            if not os.path.exists(destino):
                os.makedirs(os.path.dirname(destino), exist_ok=True)
                shutil.copy(viejo, destino)

# "siguiente": solo el próximo capítulo que falta (con los cortos que vengan antes).
# Deja un archivo _quedan-N.txt para que la app sepa cuántos faltan después de este.
if CAPITULOS.strip().lower() == "siguiente":
    hechos = {int(os.path.basename(m)[:3]) for m in glob.glob(os.path.join(SALIDA, "[0-9][0-9][0-9] - *.mp3"))}
    SOLO = set()
    for i, c in enumerate(CAPS, 1):
        if i in hechos:
            continue
        SOLO.add(i)
        if len(c["texto"]) >= 1500:
            break
    quedan = len(CAPS) - len(hechos | SOLO)
    for viejo in glob.glob(os.path.join(SALIDA, "_quedan-*.txt")):
        os.remove(viejo)
    with open(os.path.join(SALIDA, f"_quedan-{quedan}.txt"), "w") as f:
        f.write(str(quedan))
    print(f"\n➡️ Esta vez: capítulo(s) {sorted(SOLO) or 'ninguno'} · después quedan {quedan}")
    if not SOLO:
        SOLO = {0}  # nada que hacer (un conjunto vacío significaría "todos")


In [ ]:
#@title 🔧 Funciones para generar el audio
# ===== Generación del audio, capítulo por capítulo (con reanudación) =====
import time, glob, os, shutil, subprocess, zipfile, json
import numpy as np
import soundfile as sf

FRASE_MUESTRA = ("Hola. Esta es la voz que va a narrar tu audiolibro. "
                 "Si te gusta cómo suena, cambia PRUEBA a False y vuelve a ejecutar todo.")


def _mp3(wav_path, mp3_path, titulo, album, pista):
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", wav_path,
                    "-codec:a", "libmp3lame", "-b:a", "96k",
                    "-metadata", f"title={titulo}", "-metadata", f"album={album}",
                    "-metadata", f"track={pista}", mp3_path], check=True)


def _unir_partes(partes, sr, pausa_s=0.35):
    """Une los fragmentos con una pausa. Devuelve (audio, segundo de inicio de cada fragmento)."""
    silencio = np.zeros(int(sr * pausa_s), dtype=np.float32)
    trozos, inicios, muestra = [], [], 0
    for p in partes:
        a, _ = sf.read(p, dtype="float32")
        if a.ndim > 1:
            a = a.mean(axis=1)
        inicios.append(muestra / sr)
        trozos += [a, silencio]
        muestra += len(a) + len(silencio)
    return (np.concatenate(trozos) if trozos else silencio), inicios


def _guardar_sincronizacion(ruta, titulo, frags, inicios):
    """Archivo .json junto al MP3: qué frase suena en cada segundo (para seguir el texto en la app)."""
    datos = {"version": 1, "titulo": titulo,
             "frases": [{"t": round(t, 2), "texto": f} for t, f in zip(inicios, frags)]}
    with open(ruta, "w", encoding="utf-8") as fh:
        json.dump(datos, fh, ensure_ascii=False)


def generar_libro(hablar, sr, titulo_libro, capitulos, salida, trocear,
                  nombre_archivo, ref_wav, solo=None, horas_max=11.0,
                  max_chars=250, log=print):
    """hablar(texto, ref_wav) -> np.array de audio.
    Guarda cada fragmento apenas se genera, así que si la sesión se corta,
    al volver a ejecutar sigue desde el último fragmento terminado."""
    os.makedirs(salida, exist_ok=True)
    inicio = time.time()
    limite = inicio + horas_max * 3600
    total_chars = sum(len(c["texto"]) for c in capitulos)
    hechos_chars, gen_chars = 0, 0
    mp3s = []
    for n, cap in enumerate(capitulos, 1):
        if solo and n not in solo:
            # ya hecho en una corrida anterior: cuenta para estimar lo que falta del libro
            if os.path.exists(os.path.join(salida, f"{n:03d} - {nombre_archivo(cap['titulo'])}.mp3")):
                hechos_chars += len(cap["texto"])
            continue
        base = f"{n:03d} - {nombre_archivo(cap['titulo'])}"
        mp3 = os.path.join(salida, base + ".mp3")
        if os.path.exists(mp3):
            log(f"✔ {base} (ya estaba listo)")
            mp3s.append(mp3); hechos_chars += len(cap["texto"])
            continue
        partes_dir = os.path.join(salida, "_partes", f"{n:03d}")
        os.makedirs(partes_dir, exist_ok=True)
        frags = trocear(cap["texto"], max_chars)
        log(f"▶ {base}: {len(frags)} fragmentos")
        partes = []
        for i, frag in enumerate(frags):
            p = os.path.join(partes_dir, f"{i:05d}.wav")
            partes.append(p)
            if os.path.exists(p):
                hechos_chars += len(frag)
                continue
            if time.time() > limite:
                log(f"⏸ Se alcanzó el tiempo máximo ({horas_max} h). "
                    "Vuelve a ejecutar para continuar desde aquí.")
                return mp3s, False
            t0 = time.time()
            audio = hablar(frag, ref_wav)
            tmp = p + ".tmp.wav"                       # escritura segura
            sf.write(tmp, np.asarray(audio, dtype=np.float32), sr)
            os.replace(tmp, p)
            gen_chars += len(frag); hechos_chars += len(frag)
            if (i + 1) % 10 == 0 or i + 1 == len(frags):
                vel = gen_chars / max(time.time() - inicio, 1e-6)
                falta_h = (total_chars - hechos_chars) / max(vel, 1e-6) / 3600
                log(f"   {i+1}/{len(frags)}  ·  quedan ~{falta_h:.1f} h para todo el libro"
                    f"  ·  último fragmento: {time.time()-t0:.1f} s")
        wav_tmp = os.path.join("/tmp", f"cap_{n:03d}.wav")
        audio_cap, inicios = _unir_partes(partes, sr)
        sf.write(wav_tmp, audio_cap, sr)
        _mp3(wav_tmp, mp3, cap["titulo"], titulo_libro, n)
        _guardar_sincronizacion(os.path.join(salida, base + ".json"), cap["titulo"], frags, inicios)
        os.remove(wav_tmp)
        shutil.rmtree(partes_dir, ignore_errors=True)
        mp3s.append(mp3)
        log(f"✔ {base} listo")
    shutil.rmtree(os.path.join(salida, "_partes"), ignore_errors=True)
    return mp3s, True


def _duracion(path):
    r = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration",
                        "-of", "default=nw=1:nk=1", path], capture_output=True, text=True)
    return float(r.stdout.strip())


def unir_m4b(mp3s, destino, titulo_libro, titulos):
    """Une los capítulos en un solo audiolibro .m4b con marcadores de capítulo."""
    lista = "/tmp/lista.txt"
    with open(lista, "w") as f:
        for m in mp3s:
            f.write("file '" + os.path.abspath(m).replace("'", "'\\''") + "'\n")
    meta = "/tmp/meta.txt"
    with open(meta, "w") as f:
        f.write(";FFMETADATA1\ntitle=" + titulo_libro + "\n")
        t = 0
        for m, tit in zip(mp3s, titulos):
            d = int(_duracion(m) * 1000)
            f.write(f"[CHAPTER]\nTIMEBASE=1/1000\nSTART={t}\nEND={t+d}\ntitle={tit}\n")
            t += d
    tmp = "/tmp/libro.m4b"
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-f", "concat", "-safe", "0",
                    "-i", lista, "-i", meta, "-map_metadata", "1", "-map", "0:a",
                    "-c:a", "aac", "-b:a", "64k", tmp], check=True)
    shutil.copy(tmp, destino); os.remove(tmp)


def comprimir(archivos, destino):
    """Zip con los MP3 y, si existen, sus .json de sincronización con el texto."""
    with zipfile.ZipFile(destino, "w", zipfile.ZIP_STORED) as z:
        for a in archivos:
            z.write(a, os.path.basename(a))
            sinc = os.path.splitext(a)[0] + ".json"
            if os.path.exists(sinc):
                z.write(sinc, os.path.basename(sinc))


In [ ]:
#@title 🧠 Cargar VoxCPM2 y preparar la voz (la primera vez descarga ~5 GB)
import inspect, torch
from voxcpm import VoxCPM

assert torch.cuda.is_available(), ("No hay GPU. Colab: Entorno de ejecución → Cambiar tipo → GPU T4. "
                                   "Kaggle: Accelerator → GPU.")
print("GPU:", torch.cuda.get_device_name(0))

import gc
import voxcpm.model.voxcpm2 as _vx2

# Precisión: bfloat16 en GPUs modernas; float32 en la T4 (no tiene bfloat16 nativo y así es estable).
_cap = torch.cuda.get_device_capability(0)[0]
PRECISION = "bfloat16" if _cap >= 8 else "float32"
_vx2.pick_runtime_dtype = lambda device, configurado: PRECISION
print("Precisión:", PRECISION)

def _cargar(en_gpu):
    # en_gpu=True: arma el modelo directo en la GPU para no agotar la RAM de Colab (~12 GB)
    opciones = dict(load_denoiser=False, optimize=False)
    if en_gpu:
        with torch.device("cuda"):
            return VoxCPM.from_pretrained("openbmb/VoxCPM2", **opciones)
    return VoxCPM.from_pretrained("openbmb/VoxCPM2", **opciones)

if "modelo" in globals():
    # Se volvió a ejecutar sin reiniciar: reutilizar el modelo que ya está en la GPU
    # (cargarlo de nuevo agotaba la memoria de la GPU).
    print("El modelo ya estaba cargado; se reutiliza.")
else:
    try:
        modelo = _cargar(en_gpu=True)
    except torch.cuda.OutOfMemoryError:
        raise
    except Exception as e:
        print("Reintentando la carga de forma normal…", repr(e)[:200])
        gc.collect(); torch.cuda.empty_cache()
        modelo = _cargar(en_gpu=False)
gc.collect(); torch.cuda.empty_cache()
SR = modelo.tts_model.sample_rate
print(f"Modelo listo · memoria GPU usada: {torch.cuda.memory_allocated()/1e9:.1f} GB")

# (generate() solo reenvía *args/**kwargs; los nombres reales están en _generate)
_firma = inspect.signature(getattr(modelo, "_generate", modelo.generate)).parameters
_admitidos = None if any(p.kind == p.VAR_KEYWORD for p in _firma.values()) else set(_firma)
if _admitidos is not None and "reference_wav_path" not in _admitidos:
    raise RuntimeError("Esta versión de voxcpm no acepta voz de referencia.")
def hablar(texto, ref_wav=None, prefijo=""):
    opciones = dict(text=prefijo + texto, cfg_value=2.0, inference_timesteps=int(CALIDAD),
                    normalize=False,            # el normalizador de números no está hecho para español
                    retry_badcase=True, retry_badcase_max_times=3,
                    retry_badcase_ratio_threshold=6.0)
    if ref_wav:
        opciones["reference_wav_path"] = ref_wav
    if _admitidos is not None:
        opciones = {k: v for k, v in opciones.items() if k in _admitidos}
    with torch.inference_mode():
        return modelo.generate(**opciones)

# La voz se fija en un audio de referencia para que suene IGUAL en todo el libro.
VOZ_REF = os.path.join(SALIDA, "voz_referencia.wav")
if MODO_VOZ == "clonar":
    import shutil
    if RUTA_CLON.lower().endswith(".wav"):
        shutil.copy(RUTA_CLON, VOZ_REF)
    else:
        import subprocess
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", RUTA_CLON,
                        "-ac", "1", "-ar", "16000", VOZ_REF], check=True)
    print("🎙️ Usando tu voz para clonar.")
elif not os.path.exists(VOZ_REF):
    print("🎨 Creando la voz a partir de la descripción…")
    muestra = hablar(FRASE_MUESTRA, prefijo=f"({DESCRIPCION_VOZ}) ")
    sf.write(VOZ_REF, np.asarray(muestra, dtype=np.float32), SR)
else:
    print("🎨 Reutilizando la voz creada antes (borra voz_referencia.wav para crear una nueva).")

from IPython.display import Audio, display
print("Así suena la voz:")
display(Audio(VOZ_REF))


In [ ]:
#@title ▶️ Generar el audiolibro
import time
t_ini = time.time()

if PRUEBA:
    print("🧪 MODO PRUEBA: unos segundos del primer capítulo.\n")
    frags = trocear(CAPS[min(SOLO) - 1 if SOLO else 0]["texto"])[:3]
    audios = [np.asarray(hablar(f, VOZ_REF), dtype=np.float32) for f in frags]
    pausa = np.zeros(int(SR * 0.35), dtype=np.float32)
    prueba = np.concatenate([x for a in audios for x in (a, pausa)])
    ruta = os.path.join(SALIDA, "prueba.wav")
    sf.write(ruta, prueba, SR)
    seg_audio = len(prueba) / SR
    seg_trabajo = time.time() - t_ini
    display(Audio(ruta))
    total_car = sum(len(c["texto"]) for i, c in enumerate(CAPS, 1) if SOLO is None or i in SOLO)
    car_prueba = sum(len(f) for f in frags)
    est_h = seg_trabajo / car_prueba * total_car / 3600
    print(f"Generé {seg_audio:.0f} s de audio en {seg_trabajo:.0f} s.")
    print(f"Estimación para el libro completo en esta GPU: ~{est_h:.1f} horas de trabajo.")
    print("\n👉 Si te gusta, cambia PRUEBA a False y vuelve a ejecutar todo.")
else:
    mp3s, terminado = generar_libro(
        hablar, SR, TITULO_LIBRO, CAPS, SALIDA, trocear, nombre_archivo,
        VOZ_REF, solo=SOLO, horas_max=HORAS_MAXIMAS)
    if terminado and SOLO is None and UN_SOLO_ARCHIVO_M4B and mp3s:
        print("📚 Uniendo todo en un solo archivo .m4b…")
        unir_m4b(mp3s, os.path.join(SALIDA, nombre_archivo(TITULO_LIBRO) + ".m4b"),
                 TITULO_LIBRO, [c["titulo"] for c in CAPS])
    if EN_KAGGLE and mp3s:
        comprimir(mp3s, os.path.join(BASE_SALIDA, nombre_archivo(TITULO_LIBRO) + "_mp3.zip"))
    estado = "✅ ¡Listo!" if terminado else "⏸ Pausado: vuelve a ejecutar para continuar."
    print(f"\n{estado}  ({(time.time()-t_ini)/3600:.1f} h)  Archivos en: {SALIDA}")
